<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-02-embeddings-and-vectors/lesson-2.3-firestore-vector/practice/GCP_Capstone_2.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 2.3 — Firestore Vector Search

8 exercises with complete solutions. Create indexes, store embeddings, query with find_nearest(), and build a full RAG pipeline.

Runnable companion to the published practice lab. Each exercise below shows the objective and a complete solution. Cloud Shell / `gcloud` steps are `%%bash` cells; Python steps run in Colab after you authenticate and set your project.

---

## Exercise 1: Create Vector Index  
**Difficulty:** Easy

Run the gcloud commands to create the vector indexes on `chunks`: `tenant_id` ASC + `embedding` (768 dimensions) - the production index - and `tenant_id` + `doc_type` + `embedding` for the filtered exercise.

1. Open Cloud Shell
2. Run gcloud firestore indexes composite create
3. Verify with gcloud firestore operations list

**Solution:**

In [ ]:
# --- Setup: install + auth + project (run me first) ---
!pip install -q google-genai==2.21.0 google-cloud-firestore
from google.colab import auth
auth.authenticate_user()

import subprocess
PROJECT_ID = 'documind-ai-YOUR-ID'   # CHANGE THIS to your project id
!gcloud config set project {PROJECT_ID}
# enable the APIs this lab calls (idempotent, quiet). Lesson 1.1 already enabled them for the
# capstone project; on a fresh project the databases create below fails without this.
subprocess.run(['gcloud','services','enable','firestore.googleapis.com','aiplatform.googleapis.com','--project',PROJECT_ID,'--quiet'],check=False)
# ensure the Firestore (default) database exists (idempotent). COURSE RULE: asia-south1
# (Mumbai), and that location is PERMANENT - it cannot be changed after creation.
if '(default)' not in subprocess.run(['gcloud','firestore','databases','list','--project',PROJECT_ID,'--format=value(name)'],capture_output=True,text=True).stdout:
    subprocess.run(['gcloud','firestore','databases','create','--location=asia-south1','--project',PROJECT_ID],check=False)
print('Setup done')

In [ ]:
%%bash
# THE production index (Exercise 3): tenant_id ASC + embedding - the same index the kit provisions
# (deploy/terraform/firestore_indexes.tf). || true so re-runs do not fail on "already exists".
# There is deliberately NO unfiltered index: on a multi-tenant store an unfiltered query is a
# cross-tenant read, and the absence of the index is what makes it impossible.
gcloud firestore indexes composite create \
  --collection-group=chunks --query-scope=COLLECTION \
  --field-config field-path=tenant_id,order=ASCENDING \
  --field-config field-path=embedding,vector-config='{"dimension":"768","flat":"{}"}' \
  --database="(default)" || true

# Composite index tenant_id + doc_type + embedding -- needed by the FILTERED query (Exercise 5)
gcloud firestore indexes composite create \
  --collection-group=chunks --query-scope=COLLECTION \
  --field-config field-path=tenant_id,order=ASCENDING \
  --field-config field-path=doc_type,order=ASCENDING \
  --field-config field-path=embedding,vector-config='{"dimension":"768","flat":"{}"}' \
  --database="(default)" || true

# Check status
gcloud firestore operations list --database="(default)"

### Wait for the vector index to build (run after Setup)
This blocks until the Firestore vector index reports `READY` — about 2-5 minutes on the **first** run only (returns instantly afterwards). The retrieval cells below raise `FAILED_PRECONDITION` until this prints READY. Safe to re-run any time.

In [ ]:
%%bash
# Block until every index is READY (find_nearest raises FAILED_PRECONDITION until then; ~2-5 min)
echo "Waiting for vector indexes to reach READY..."
for i in $(seq 1 40); do
  STATES=$(gcloud firestore indexes composite list --database="(default)" --format='value(state)')
  if [ -n "$STATES" ] && ! echo "$STATES" | grep -q CREATING; then echo "Indexes READY"; break; fi
  sleep 15
done

## Exercise 2: Store 5 Documents with Embeddings  
**Difficulty:** Easy

Write 5 documents to Firestore, each with text content, a 768-dim embedding via Vector(), and a category field.

1. Initialize Firestore and genai clients
2. Embed 5 texts with RETRIEVAL_DOCUMENT
3. Store each with Vector(embedding)
4. Verify in Firestore Console

**Solution:**

In [ ]:
from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google import genai
from google.genai import types

db = firestore.Client(project="documind-ai-YOUR-ID")
ai = genai.Client(enterprise=True, project="documind-ai-YOUR-ID", location="us-central1")
TENANT_ID = "acme"                      # the teaching tenant; every query is filtered by it
col = db.collection("chunks")           # ONE collection, from here to production

texts = [
    ("ai_ml", "Neural networks learn from labeled data."),
    ("gcp", "Cloud Run deploys containers serverlessly."),
    ("india", "Hyderabad is a major tech hub in India."),
    ("python", "Python list comprehensions are concise."),
    ("ai_ml", "RAG combines retrieval with generation."),
]

# text-embedding-005 takes up to 250 texts per request (2.2's batch rule); one per call keeps the loop readable
for i, (cat, text) in enumerate(texts):
    emb = ai.models.embed_content(
        model="text-embedding-005", contents=text,
        config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":768}
    ).embeddings[0]
    # The CANONICAL document - the six field names services/ingest/indexer.py writes in production
    col.document(f"test_{i}").set({
        "tenant_id": TENANT_ID, "text": text,
        "source_uri": f"gs://documind-ai-YOUR-ID-uploads/{TENANT_ID}/{cat}/test_{i}.md",
        "page_start": None, "doc_type": cat, "embedding": Vector(emb.values)})
print(f"Stored {len(texts)} documents")

## Exercise 3: First find_nearest() Query  
**Difficulty:** Easy

Run your first vector search. Print top-3 results with cosine similarity scores.

1. Embed a query with RETRIEVAL_QUERY
2. Call find_nearest() with COSINE distance
3. Convert distance to similarity (1 - distance)
4. Print ranked results

**Solution:**

In [ ]:
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud.firestore_v1.base_query import FieldFilter

query = "How does deep learning work?"
q_emb = ai.models.embed_content(
    model="text-embedding-005", contents=query,
    config={"task_type":"RETRIEVAL_QUERY","output_dimensionality":768}
).embeddings[0].values

results = col.where(filter=FieldFilter("tenant_id", "==", TENANT_ID)).find_nearest(
    vector_field="embedding",
    query_vector=Vector(q_emb),
    distance_measure=DistanceMeasure.COSINE,
    limit=3,
    distance_result_field="vector_distance",
).get()

print(f"Query: {query}")
for doc in results:
    d = doc.to_dict()
    sim = 1 - d["vector_distance"]
    print(f"  [{sim:.4f}] {d['text']}")

## Exercise 4: Batch Ingest 20 Documents  
**Difficulty:** Medium

Load the full 20-document corpus: one embedding request per text, one batch write. Verify all are searchable.

1. Define 20 documents across 4 categories
2. Embed them with text-embedding-005 (up to 250 texts per request; one per call keeps the loop readable)
3. Batch write all 20 to Firestore
4. Run a test query to verify

**Solution:**

In [ ]:
# Self-contained: the corpus is defined right here, nothing to import from elsewhere.
corpus = [
    {"text": "Neural networks learn patterns from labeled training data.", "doc_type": "ai_ml"},
    {"text": "Transformers use self-attention to process sequences in parallel.", "doc_type": "ai_ml"},
    {"text": "RAG combines document retrieval with language model generation.", "doc_type": "ai_ml"},
    {"text": "Fine-tuning adapts a pre-trained model to a specific domain.", "doc_type": "ai_ml"},
    {"text": "Embeddings convert text into dense numerical vectors.", "doc_type": "ai_ml"},
    {"text": "Cloud Run deploys containerized applications serverlessly.", "doc_type": "gcp"},
    {"text": "BigQuery processes petabytes of data using SQL queries.", "doc_type": "gcp"},
    {"text": "Firestore is a NoSQL document database with real-time sync.", "doc_type": "gcp"},
    {"text": "Vertex AI provides managed infrastructure for ML workflows.", "doc_type": "gcp"},
    {"text": "Cloud Storage offers durable object storage at low cost.", "doc_type": "gcp"},
    {"text": "Hyderabad is the capital of Telangana and a major tech hub.", "doc_type": "india"},
    {"text": "India Digital Public Infrastructure includes UPI and Aadhaar.", "doc_type": "india"},
    {"text": "DPDP Act 2023 governs personal data protection in India.", "doc_type": "india"},
    {"text": "IndiaAI Mission allocates 10,000 GPUs for AI research.", "doc_type": "india"},
    {"text": "Bangalore, Hyderabad, and Pune are top Indian tech cities.", "doc_type": "india"},
    {"text": "Python list comprehensions provide concise filtering syntax.", "doc_type": "python"},
    {"text": "Asyncio enables concurrent I/O-bound operations in Python.", "doc_type": "python"},
    {"text": "Type hints improve code readability and enable analysis.", "doc_type": "python"},
    {"text": "Virtual environments isolate project dependencies.", "doc_type": "python"},
    {"text": "Pydantic validates data using Python type annotations.", "doc_type": "python"},
]

# text-embedding-005 takes up to 250 texts per request (2.2's batch rule); one per call keeps the loop readable
batch = db.batch()
for i, doc_data in enumerate(corpus):
    emb = ai.models.embed_content(
        model="text-embedding-005", contents=doc_data["text"],
        config={"task_type":"RETRIEVAL_DOCUMENT","output_dimensionality":768}
    ).embeddings[0].values
    batch.set(col.document(f"doc_{i:03d}"), {
        **doc_data, "tenant_id": TENANT_ID,
        "source_uri": f"gs://documind-ai-YOUR-ID-uploads/{TENANT_ID}/{doc_data['doc_type']}/doc_{i:03d}.md",
        "page_start": None, "embedding": Vector(emb)})
    print(f"  embedded {i+1}/{len(corpus)}", end="\r")
batch.commit()
print(f"Loaded {len(corpus)} documents")

## Exercise 5: Filtered Vector Search  
**Difficulty:** Medium

Use the composite index tenant_id + doc_type + embedding. Search within "ai_ml" only. Compare with the tenant-only results.

1. Create composite index via gcloud
2. Run the tenant-only find_nearest()
3. Run .where(tenant_id).where(filter=FieldFilter("doc_type","==","ai_ml")).find_nearest()
4. Compare results and relevance

**Solution:**

In [ ]:
from google.cloud.firestore_v1.base_query import FieldFilter

# Tenant-only: the predicate every DocuMind query carries
tenant_only = col.where(filter=FieldFilter("tenant_id", "==", TENANT_ID)).find_nearest(
    vector_field="embedding", query_vector=Vector(q_emb),
    distance_measure=DistanceMeasure.COSINE, limit=3,
    distance_result_field="dist").get()

# Tenant + doc_type (requires the composite index; positional .where("doc_type","==",...) is deprecated)
filtered = (col.where(filter=FieldFilter("tenant_id", "==", TENANT_ID))
               .where(filter=FieldFilter("doc_type", "==", "ai_ml"))).find_nearest(
    vector_field="embedding", query_vector=Vector(q_emb),
    distance_measure=DistanceMeasure.COSINE, limit=3,
    distance_result_field="dist").get()

print("Tenant only:")
for d in tenant_only: print(f"  [{1-d.to_dict()['dist']:.4f}] {d.to_dict()['text'][:50]}")
print("\nFiltered (ai_ml only):")
for d in filtered: print(f"  [{1-d.to_dict()['dist']:.4f}] {d.to_dict()['text'][:50]}")

## Exercise 6: Distance Threshold Testing  
**Difficulty:** Medium

Set distance_threshold=0.3. Test with relevant and irrelevant queries. Verify threshold filters correctly.

1. Query with a relevant question (should return results)
2. Query with an irrelevant question (should return empty)
3. Compare result counts

**Solution:**

In [ ]:
for query in ["How do neural networks learn?", "What is the recipe for chocolate cake?"]:
    q = ai.models.embed_content(
        model="text-embedding-005", contents=query,
        config={"task_type":"RETRIEVAL_QUERY","output_dimensionality":768}
    ).embeddings[0].values
    results = col.where(filter=FieldFilter("tenant_id", "==", TENANT_ID)).find_nearest(
        vector_field="embedding", query_vector=Vector(q),
        distance_measure=DistanceMeasure.COSINE, limit=5,
        distance_threshold=0.3,  # similarity > 0.7
    ).get()
    print(f"Query: {query[:40]}... → {len(results)} results")

## Exercise 7: Full RAG Pipeline  
**Difficulty:** Challenge

Build rag_query() that embeds, searches Firestore, and generates with Gemini. Test across 4 categories.

1. Build function: embed → find_nearest → generate
2. Test: "How do transformers work?"
3. Test: "What is India's data protection law?" (filtered)
4. Test: "Best practices for Python async?"

**Solution:**

In [ ]:
from google.cloud.firestore_v1.base_query import FieldFilter
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure

# Two clients: embeddings are regional-only (us-central1), Gemini 3.x generation is global-only.
gen = genai.Client(enterprise=True, project="documind-ai-YOUR-ID", location="global")

def rag_query(question, doc_type=None, top_k=3):
    q_emb = ai.models.embed_content(
        model="text-embedding-005", contents=question,
        config={"task_type":"RETRIEVAL_QUERY","output_dimensionality":768}
    ).embeddings[0].values

    ref = col.where(filter=FieldFilter("tenant_id", "==", TENANT_ID))   # always this tenant
    if doc_type:
        ref = ref.where(filter=FieldFilter("doc_type", "==", doc_type))
    docs = ref.find_nearest(
        vector_field="embedding", query_vector=Vector(q_emb),
        distance_measure=DistanceMeasure.COSINE,
        limit=top_k, distance_result_field="dist",
        distance_threshold=0.5,
    ).get()

    context = "\n".join(
        f"[Source {n}] {d.to_dict()['text']}" for n, d in enumerate(docs, 1))
    response = gen.models.generate_content(
        model="gemini-3.6-flash",
        contents=f"Answer using ONLY the context.\n\nContext:\n{context}\n\nQuestion: {question}",
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0)))

    print(f"Query: {question}")
    print(f"Sources: {len(docs)} documents")
    for n, d in enumerate(docs, 1):
        dd = d.to_dict()
        print(f"  [Source {n}] [{1-dd['dist']:.4f}] {dd['text'][:60]}...")
    print(f"\nAnswer: {response.text}\n")
    return response.text

rag_query("How do transformers work?")
rag_query("What is India's data protection law?", doc_type="india")
rag_query("Best practices for Python async?")

## Exercise 8: FirestoreRAG Production Module  
**Difficulty:** Challenge

Build the complete FirestoreRAG class with embed(), ingest(), search(), query() methods.

1. Define class with __init__ for clients
2. Implement embed() with configurable task type
3. Implement search() with optional category filter
4. Implement query() for full RAG
5. Test end-to-end with 20 documents

**Solution:**

In [ ]:
PROJECT_ID = "documind-ai-YOUR-ID"
CHUNKS_COLLECTION = "chunks"   # ONE collection for the whole course, 2.3 -> 4.x -> Module 12
TENANT_ID = "acme"

from google.cloud import firestore
from google.cloud.firestore_v1.vector import Vector
from google.cloud.firestore_v1.base_vector_query import DistanceMeasure
from google.cloud.firestore_v1.base_query import FieldFilter
from google import genai
from google.genai import types

EMBED_MODEL = "text-embedding-005"   # ONE model per store
GEN_MODEL = "gemini-3.6-flash"
DIMS = 768

class FirestoreRAG:
    def __init__(self, project, collection_name=CHUNKS_COLLECTION, tenant_id=TENANT_ID):
        self.db = firestore.Client(project=project)
        self.ai = genai.Client(enterprise=True, project=project, location="us-central1")  # embeddings: regional-only
        self.gen = genai.Client(enterprise=True, project=project, location="global")  # generation: global endpoint
        self.collection = self.db.collection(collection_name)
        self.tenant_id = tenant_id

    def embed(self, text, task="RETRIEVAL_DOCUMENT"):
        r = self.ai.models.embed_content(
            model=EMBED_MODEL, contents=text,
            config=types.EmbedContentConfig(task_type=task, output_dimensionality=DIMS))
        return r.embeddings[0].values

    def ingest(self, documents):
        """Ingest list of dicts with 'text' and optional 'doc_type'. Writes the canonical
        document - the shape services/ingest/indexer.py writes in production."""
        batch = self.db.batch()
        pending = 0
        for i, doc in enumerate(documents):
            vec = self.embed(doc["text"], task="RETRIEVAL_DOCUMENT")
            ref = self.collection.document(f"doc_{i:04d}")
            batch.set(ref, {"tenant_id": self.tenant_id, "text": doc["text"],
                            "source_uri": doc.get("source_uri", f"gs://{PROJECT_ID}-uploads/{self.tenant_id}/doc_{i:04d}.md"),
                            "page_start": doc.get("page_start"), "doc_type": doc.get("doc_type", "policy"),
                            "embedding": Vector(vec)})
            pending += 1
            if pending == 500:  # Firestore commits at most 500 writes per batch
                batch.commit()
                batch = self.db.batch()
                pending = 0
            print(f"  embedded {i+1}/{len(documents)}", end="\r")
        if pending:
            batch.commit()
        return len(documents)

    def search(self, query, category=None, top_k=5, threshold=0.5):
        q_vec = self.embed(query, task="RETRIEVAL_QUERY")
        ref = self.collection.where(filter=FieldFilter('tenant_id', '==', self.tenant_id))
        if category:
            ref = ref.where(filter=FieldFilter("doc_type", "==", category))
        docs = ref.find_nearest(
            vector_field="embedding", query_vector=Vector(q_vec),
            distance_measure=DistanceMeasure.COSINE,
            limit=top_k, distance_result_field="dist",
            distance_threshold=threshold).get()
        return [{"chunk_id": d.id, "text": d.to_dict()["text"],
                 "source_uri": d.to_dict().get("source_uri", ""),
                 "page_start": d.to_dict().get("page_start"),
                 "doc_type": d.to_dict().get("doc_type", ""),
                 "similarity": 1 - d.to_dict()["dist"]} for d in docs]

    def query(self, question, category=None):
        results = self.search(question, category)
        context = "\n".join(
            f"[Source {n}] {r['text']}" for n, r in enumerate(results, 1))
        response = self.gen.models.generate_content(
            model=GEN_MODEL,
            contents=f"Context:\n{context}\n\nQuestion: {question}",
            config=types.GenerateContentConfig(thinking_config=types.ThinkingConfig(thinking_budget=0)))
        return {"answer": response.text, "sources": results}

# Test: query() end-to-end against the 20 documents loaded in Cell 3
rag = FirestoreRAG(PROJECT_ID)
out = rag.query('What is RAG?')
print(out['answer'])
for s in out['sources']:
    print(f"  [{s['similarity']:.4f}] {s['text']}")
